# 01 — KuaiSearch-Lite preprocessing và embedding

Notebook chạy trên Kaggle, tải ba phần `rank_lite`, `items_lite`, `users_lite`, preprocess bằng Pandas và tạo product embedding đầu vào cho RQ-VAE.

## 0. Cấu hình

In [ ]:
from pathlib import Path

RAW_ROOT = None
OUTPUT_ROOT = None

VALIDATION_PERCENT = 10
SEED = 2026
RESET_OUTPUT = True
DOWNLOAD_IF_MISSING = True

EMBEDDING_MODEL = "jinaai/jina-embeddings-v5-text-nano-clustering"
EMBEDDING_DIM = 256
EMBEDDING_BATCH_SIZE = 128

## 1. Import

In [ ]:
import json
import os
import shutil
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub>=0.27", "pyarrow>=14",
    "sentence-transformers>=5.2.0", "transformers>=5.1.0", "peft>=0.15.2",
    "scikit-learn>=1.4",
])

import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

print("Pandas:", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")

## 2. Tìm hoặc tải KuaiSearch-Lite

Notebook ưu tiên dữ liệu đã attach trong `/kaggle/input`. Nếu chưa có, bật Internet để tải từ Hugging Face. `HF_TOKEN` là tùy chọn.

In [ ]:
HF_REPO_ID = "benchen4395/KuaiSearch"
RAW_FILES = {
    "ranking": Path("rank_lite/train.jsonl"),
    "items": Path("items_lite/train.jsonl"),
    "users": Path("users_lite/train.jsonl"),
}


def has_raw_files(root):
    return root is not None and all((Path(root) / path).is_file() for path in RAW_FILES.values())


if RAW_ROOT is not None:
    RAW_ROOT = Path(RAW_ROOT).expanduser().resolve()
elif Path("/kaggle/input").exists():
    matches = list(Path("/kaggle/input").glob("**/rank_lite/train.jsonl"))
    RAW_ROOT = matches[0].parents[1] if matches else None

if not has_raw_files(RAW_ROOT):
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError("KuaiSearch-Lite was not found.")
    RAW_ROOT = (
        Path("/kaggle/working/kuaisearch-raw")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "kuaisearch-raw"
    )
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        local_dir=str(RAW_ROOT),
        allow_patterns=[path.as_posix() for path in RAW_FILES.values()],
        token=os.environ.get("HF_TOKEN"),
    )

if not has_raw_files(RAW_ROOT):
    raise FileNotFoundError("KuaiSearch-Lite download is incomplete.")

if OUTPUT_ROOT is None:
    OUTPUT_ROOT = (
        Path("/kaggle/working/preprocessed")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "preprocessed"
    )
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()

if OUTPUT_ROOT.exists() and RESET_OUTPUT:
    if OUTPUT_ROOT.name != "preprocessed":
        raise ValueError(f"Refusing to reset unsafe path: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("RAW_ROOT:", RAW_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## 3. Đọc dữ liệu

In [ ]:
items = pd.read_json(RAW_ROOT / RAW_FILES["items"], lines=True)
users = pd.read_json(RAW_ROOT / RAW_FILES["users"], lines=True)
ranking = pd.read_json(RAW_ROOT / RAW_FILES["ranking"], lines=True)

print(f"Items: {len(items):,}")
print(f"Users: {len(users):,}")
print(f"Ranking rows: {len(ranking):,}")
print(ranking["split"].value_counts())

## 4. Product catalog

`normalized_text` ghép title, brand, seller và ba cấp category để dùng ở bước embedding.

In [ ]:
items = items.rename(columns={"item_id": "product_id"})
if not items["product_id"].is_unique:
    raise ValueError("Expected one row per product_id in items_lite.")
text_columns = [
    "item_title", "brand_name", "seller_name",
    "category_level1_name", "category_level2_name", "category_level3_name",
]
items[text_columns] = items[text_columns].fillna("").astype(str)
items.insert(0, "product_index", range(len(items)))
items["normalized_text"] = (
    "title: " + items["item_title"]
    + " | brand: " + items["brand_name"]
    + " | seller: " + items["seller_name"]
    + " | category_l1: " + items["category_level1_name"]
    + " | category_l2: " + items["category_level2_name"]
    + " | category_l3: " + items["category_level3_name"]
)

items.to_parquet(OUTPUT_ROOT / "items.parquet", index=False, compression="zstd")

display(items.head(3))

## 5. User profiles

In [ ]:
user_columns = [
    "user_id", "gender", "age_bucket",
    "fre_country", "fre_province", "fre_city",
]
users = users[user_columns]
users.to_parquet(OUTPUT_ROOT / "users.parquet", index=False, compression="zstd")

display(users.head(3))

## 6. Ranking

Hai dictionary thống kê được mở thành các cột phẳng. `split=test` của KuaiSearch được giữ làm test; các dòng còn lại được chia train/validation bằng `train_test_split`.

In [ ]:
user_stats = pd.json_normalize(ranking["user_statistical_features"]).add_prefix("user_stat_")
item_stats = pd.json_normalize(ranking["target_item_statistical_features"]).add_prefix("item_stat_")
ranking = ranking.drop(columns=[
    "user_statistical_features",
    "target_item_statistical_features",
]).join(user_stats).join(item_stats)

ranking_test = ranking[ranking["split"] == "test"].copy()
ranking_train = ranking[ranking["split"] != "test"].copy()
ranking_train, ranking_validation = train_test_split(
    ranking_train,
    test_size=VALIDATION_PERCENT / 100,
    random_state=SEED,
)

ranking_train.to_parquet(OUTPUT_ROOT / "ranking_train.parquet", index=False, compression="zstd")
ranking_validation.to_parquet(OUTPUT_ROOT / "ranking_validation.parquet", index=False, compression="zstd")
ranking_test.to_parquet(OUTPUT_ROOT / "ranking_test.parquet", index=False, compression="zstd")

print("train:", len(ranking_train))
print("validation:", len(ranking_validation))
print("test:", len(ranking_test))

## 7. Product embedding

Encode `normalized_text` bằng Jina Embeddings, chuẩn hoá L2 và lưu ma trận float16 theo đúng thứ tự `product_index`. Hai file đầu ra được `train_rqvae.py` đọc trực tiếp.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator on Kaggle before creating embeddings.")

model = SentenceTransformer(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    device="cuda",
    model_kwargs={"dtype": torch.float16},
)

embeddings = model.encode(
    items["normalized_text"].tolist(),
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    truncate_dim=EMBEDDING_DIM,
)
embeddings = np.asarray(embeddings, dtype=np.float16)

if embeddings.shape != (len(items), EMBEDDING_DIM):
    raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")
if not np.isfinite(embeddings).all():
    raise ValueError("Embeddings contain NaN or Inf.")

np.save(OUTPUT_ROOT / "global_product_embeddings.f16.npy", embeddings)
items[["product_index", "product_id"]].to_parquet(
    OUTPUT_ROOT / "global_embedding_index.parquet", index=False, compression="zstd"
)

sample = embeddings[:min(10_000, len(embeddings))].astype(np.float32)
print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Mean L2 norm:", np.linalg.norm(sample, axis=1).mean())

## 8. Manifest và kiểm tra output

In [ ]:
manifest = {
    "contract_version": "kuaisearch-lite-ranking-v1",
    "source": {"repo_id": HF_REPO_ID, "raw_root": str(RAW_ROOT)},
    "configuration": {
        "validation_percent": VALIDATION_PERCENT,
        "seed": SEED,
    },
    "embedding": {
        "model": EMBEDDING_MODEL,
        "dimension": EMBEDDING_DIM,
        "dtype": "float16",
        "normalized": True,
    },
    "products": len(items),
    "users": len(users),
    "ranking": {
        "train": len(ranking_train),
        "validation": len(ranking_validation),
        "test": len(ranking_test),
    },
}
(OUTPUT_ROOT / "preprocessing_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(ranking_train.head(3))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Preprocessing complete:", OUTPUT_ROOT)

## Artifact đầu ra

- Product catalog: `items.parquet`.
- Product embedding: `global_product_embeddings.f16.npy`, `global_embedding_index.parquet`.
- Teacher view: `ranking_train.parquet`, `ranking_validation.parquet`, `ranking_test.parquet`.
- User features: `users.parquet`.